# Assignment 01
## Convolve

In [ ]:
def convol_with_Toeplitz_matrix(img, kernel):
    """
        The function you need to implement for Q1 b).
        Inputs:
            img: array(float) 6*6
            kernel: array(float) 3*3
        Outputs:
            output: array(float)
    """
    # 1. zero padding (此时 img 是 6x6, padding 后是 8x8)
    padding_img = padding(img, 1, "zeroPadding")
    
    H, W = img.shape             # 6, 6
    pH, pW = padding_img.shape   # 8, 8
    kH, kW = kernel.shape        # 3, 3

    # build the Toeplitz matrix and compute convolution
    
    T = np.zeros((H * W, pH * pW))
    
    # --- 构建 Toeplitz 矩阵 ---
    # 1. 找到每一个 3x3 感受野在 8x8 图像中的“起始一维索引”
    row_idx_start = np.arange(H)[:, None] 
    col_idx_start = np.arange(W)
    start_idx = (row_idx_start * pW + col_idx_start).flatten() # Shape: (36,)

这段代码是 NumPy 中非常经典的操作，常用于**将二维网格坐标映射为一维索引**（例如在图像处理中提取 Patch，或将矩阵展平前计算位置）。

我们设定 $H=6, W=6$，并设定 $pW=10$（方便计算演示）。

---

### 1. `np.arange` 函数详解

`np.arange` 是 NumPy 中最基础的函数之一，用于生成等差数列数组。

*   **功能**：返回一个均匀间隔的数组值。
*   **语法**：`np.arange([start, ]stop, [step, ]dtype=None)`
    *   `start`: 起始值（包含），默认为 0。
    *   `stop`: 结束值（**不包含**）。
    *   `step`: 步长，默认为 1。
*   **在本代码中的应用**：
    ```python
    np.arange(H)  # 生成 [0, 1, 2, ..., H-1]
    np.arange(W)  # 生成 [0, 1, 2, ..., W-1]
    ```
    如果 $H=6$，`np.arange(6)` 的结果是：
    ```python
    array([0, 1, 2, 3, 4, 5])
    ```
    **形状（Shape）**：这是一个一维数组，形状为 `(6,)`。

---

### 2. 维度变换：`[:, None]`

代码的第一行有一个关键操作：
```python
row_idx_start = np.arange(H)[:, None]
```

*   **`None` 的作用**：在 NumPy 中，`None` 是 `np.newaxis` 的别名。它的作用是**增加一个维度**。
*   **切片 `[:, None]` 的含义**：
    *   `:` 表示保留第一维的所有元素。
    *   `None` 表示在第二维的位置插入一个新轴。
*   **形状变化**：
    *   原始 `np.arange(H)` 形状：`(H,)` $\rightarrow$ 例如 `(6,)`。
    *   添加 `None` 后 `row_idx_start` 形状：`(H, 1)` $\rightarrow$ 例如 `(6, 1)`。
*   **视觉变化**：
    从 **行向量** 变成了 **列向量**。
    ```python
    # 原始 (6,)
    [0, 1, 2, 3, 4, 5]

    # 变换后 (6, 1)
    [[0],
     [1],
     [2],
     [3],
     [4],
     [5]]
    ```
    **为什么要这样做？** 这是为了触发后续的**广播机制**。

---

### 3. 核心机制：NumPy 广播 (Broadcasting)

这是这段代码最难也最重要的部分。
```python
start_idx = (row_idx_start * pW + col_idx_start)
```
这里涉及两个数组的运算：
1.  `row_idx_start`: 形状 `(H, 1)`，例如 `(6, 1)`。
2.  `col_idx_start`: 形状 `(W,)`，例如 `(6,)`。

#### 3.1 广播的规则
当两个数组形状不同但进行算术运算时，NumPy 会尝试“广播”它们，使形状兼容。规则如下：
1.  **右对齐**：将两个数组的形状从后向前（从右向左）对齐。
2.  **维度兼容**：对于每一对维度，如果它们**相等**，或者其中**一个是 1**，则兼容。
3.  **扩展**：维度为 1 的数组会沿着该维度复制，以匹配另一个数组的形状。

#### 3.2 在本代码中的推导
让我们看看形状是如何匹配的：

*   `row_idx_start`: 形状 `(6, 1)`
*   `col_idx_start`: 形状 `(6,)`

**第一步：右对齐补维**
NumPy 会在 `col_idx_start` 的左侧补 1，使其维度数与 `row_idx_start` 一致。
*   `row`: `(6, 1)`
*   `col`: `(1, 6)`  <-- 隐式变换

**第二步：检查兼容性**
*   第 1 维（行）：`6` 和 `1` $\rightarrow$ 兼容（1 可以广播为 6）。
*   第 2 维（列）：`1` 和 `6` $\rightarrow$ 兼容（1 可以广播为 6）。
*   **结果形状**：`(6, 6)`。

**第三步：实际计算（外和逻辑）**
这实际上是一个**外积/外和**操作。
*   `row_idx_start` 中的每一行数据相同（因为列维是 1，被广播复制了）。
*   `col_idx_start` 中的每一列数据相同（因为行维是 1，被广播复制了）。

**具体数值演示 (假设 H=3, W=3, pW=10 以便观察)：**

```python
row_idx_start (3, 1):      col_idx_start (3,):
[[0],                      [0, 1, 2]
 [1],          
 [2]]          
```

**运算 `row_idx_start * 10 + col_idx_start`：**

1.  **乘法广播** (`row * 10`)：
    ```
    [[0],      * 10  ->   [[0],
     [1],                 [10],
     [2]]                 [20]]
    ```
2.  **加法广播** (`+ col`)：
    NumPy 会将 `[[0], [10], [20]]` 与 `[0, 1, 2]` 相加。
    *   第 0 行：`0 + [0, 1, 2]` $\rightarrow$ `[0, 1, 2]`
    *   第 1 行：`10 + [0, 1, 2]` $\rightarrow$ `[10, 11, 12]`
    *   第 2 行：`20 + [0, 1, 2]` $\rightarrow$ `[20, 21, 22]`

**最终结果矩阵 (3, 3)：**
```
[[ 0,  1,  2],
 [10, 11, 12],
 [20, 21, 22]]
```
*物理含义*：这代表了二维网格中每个点的线性索引值（行优先）。公式为 $Index = row \times Width + col$。

---

### 4. `.flatten()` 展平

```python
start_idx = (...).flatten()
```

*   **功能**：将多维数组压缩成一维数组。
*   **顺序**：默认是 'C' 顺序（行优先），即先遍历第一行，再遍历第二行...
*   **接上面的例子**：
    矩阵：
    ```
    [[ 0,  1,  2],
     [10, 11, 12],
     [20, 21, 22]]
    ```
    展平后：
    ```
    [0, 1, 2, 10, 11, 12, 20, 21, 22]
    ```
*   **对应原代码**：
    原代码注释说 `# Shape: (36,)`。这意味着经过 `flatten()` 后，原本 $(H, W)$ 的二维矩阵变成了长度为 $H \times W$ 的一维向量。

---

这段代码的整体逻辑是**生成一个二维平面的线性索引映射表**。

1.  **`np.arange(H)[:, None]`**: 生成行坐标列向量 $\begin{bmatrix} 0 \\ 1 \\ \vdots \end{bmatrix}$。
2.  **`np.arange(W)`**: 生成列坐标行向量 $\begin{bmatrix} 0 & 1 & \cdots \end{bmatrix}$。
3.  **广播运算 `* pW +`**: 利用广播机制，无需循环，一次性计算出所有 $(row, col)$ 位置对应的一维索引值 $row \times pW + col$。
    *   这里 `pW` 通常代表“步长”或“宽度”。如果这是为了将图像分块，`pW` 可能是原图的宽度。
4.  **`flatten()`**: 将计算好的索引矩阵拉直，方便后续直接用于数组索引操作。

**优势**：
*   **向量化（Vectorization）**：完全避免了 Python 的 `for` 循环，利用 C 语言底层的 NumPy 优化，速度极快。
*   **内存效率**：`arange` 生成的是视图或紧凑数组，广播在计算时才隐式扩展，不需要手动创建巨大的中间网格矩阵（如 `np.meshgrid` 有时会占用更多内存，虽然 `meshgrid` 更易读，但此写法更紧凑）。


### 5. `np.meshgrid`

```python
start_i, start_j = np.meshgrid(rows, cols, indexing='ij')
```

`meshgrid` 的作用是将这两个一维向量“编织”成两个二维矩阵。
*   `start_i`: 存储每个位置的 **行索引 (i)**。
*   `start_j`: 存储每个位置的 **列索引 (j)**。

#### 5.1 关键参数：`indexing='ij'`
这是最容易出错的地方。NumPy 支持两种索引模式：
*   **`'ij'` (矩阵索引)**：第一个轴是行 (i)，第二个轴是列 (j)。**适合图像处理、矩阵运算。**
*   **`'xy'` (笛卡尔索引)**：第一个轴是 x (列)，第二个轴是 y (行)。**适合绘图 (matplotlib)。**

**在本代码中，使用 `indexing='ij'` 意味着：**
*   `start_i` 的形状是 `(out_H, out_W)` $\rightarrow$ `(3, 4)`。
*   `start_i` 的值在 **行方向** 变化，在 **列方向** 保持不变。
*   `start_j` 的形状是 `(out_H, out_W)` $\rightarrow$ `(3, 4)`。
*   `start_j` 的值在 **列方向** 变化，在 **行方向** 保持不变。

#### 5.2 可视化结果 (未 flatten 前)

让我们看看生成的矩阵长什么样：

```python
# start_i (行索引矩阵)
# 每一行的值相同，随着行号增加而增加
[[0, 0, 0, 0],   # 第 0 行，所有列的行号都是 0
 [1, 1, 1, 1],   # 第 1 行，所有列的行号都是 1
 [2, 2, 2, 2]]   # 第 2 行，所有列的行号都是 2

# start_j (列索引矩阵)
# 每一列的值相同，随着列号增加而增加
[[0, 1, 2, 3],   # 第 0 行，列号从 0 变到 3
 [0, 1, 2, 3],   # 第 1 行，列号从 0 变到 3
 [0, 1, 2, 3]]   # 第 2 行，列号从 0 变到 3
```

**物理含义**：
如果你想获取平面上坐标 $(1, 2)$ （第 1 行，第 2 列）的索引值：
*   行索引：`start_i[1, 2]` $\rightarrow$ `1`
*   列索引：`start_j[1, 2]` $\rightarrow$ `2`
完美对应。

---

#### 5.3 对比：`meshgrid` vs 之前的广播机制

之前的代码（`arange[:, None] + arange`）和这段 `meshgrid` 代码，目的非常相似，都是生成网格坐标，但侧重点不同。

| 特性 | 广播机制 (`arange[:, None]`) | `np.meshgrid` |
| :--- | :--- | :--- |
| **代码示例** | `row = np.arange(H)[:, None]` | `row, col = np.meshgrid(..., indexing='ij')` |
| **主要用途** | **数学计算** (如计算索引偏移、距离矩阵) | **坐标生成** (明确需要 (x, y) 对时) |
| **内存占用** | 较低 (利用广播隐式扩展) | 较高 (显式生成两个完整的 (H, W) 矩阵) |
| **可读性** | 对新手较难理解 (需懂广播) | **语义清晰** (一看就知道是生成网格) |
| **输出形式** | 通常直接用于计算，不一定要展平 | 常展平后用于索引配对 |

**在本代码的上下文中：**
使用 `meshgrid` 是为了**显式地获得所有 (行，列) 坐标对**。
*   如果你只是算一个线性索引（如上一段代码 `start_idx`），用广播法 `row * W + col` 更高效。
*   如果你需要分别操作行和列（例如计算相对位置 `start_i - kernel_center_i`），用 `meshgrid` 更直观。

## Canny Edge Detector
### 为什么 Sobel 滤波器能算出梯度？

在高等数学里，求导数 $\frac{df(x)}{dx}$ 是求一个无穷小范围内的变化率。
但是在计算机的图像里，像素是一个一个的格子，没有“无穷小”，最小的距离就是 **1 个像素**。

那么在图像里，怎么求“变化率”呢？答案极其简单：**直接相减**。

**1. 降维打击：先看一维的情况**
假设有一行像素，亮度值是这样的（从黑变白）：
`[10, 10, 10, 90, 90, 90]`
肉眼可见，边缘在 `10` 和 `90` 之间。让计算机怎么找？
我们用一个非常简单的数学算子：`[-1, 0, 1]` 去滑动相乘并求和（这就叫一维卷积）：
* 放在全黑区域 `[-1*10 + 0*10 + 1*10] = 0` （变化率为0）
* 放在边缘地带 `[-1*10 + 0*10 + 1*90] = 80` （变化率极大，发现边缘！）

**2. 升级到二维：Sobel 滤波器**
你在代码里用的 Sobel X 滤波器长这样：
```text
[-1,  0,  1]
[-2,  0,  2]
[-1,  0,  1]
```
当你把这个 $3 \times 3$ 的矩阵盖在图像上的某个像素时，它的计算逻辑其实是：
**（右边那一列的像素之和） 减去 （左边那一列的像素之和）**。
* 如果右边比左边亮，得出一个正数（正梯度）。
* 如果左边和右边一样亮，结果是 0（没有边缘）。

**为什么中间那行是 2 和 -2 呢？**
因为图像常常有噪点。如果我们只看一行，很容易被噪点骗了。Sobel 聪明的地方在于，它不仅看了中心像素的左右，还看了它上下两行的左右，并且给中心那行更大的权重（2）。这就相当于**“在计算水平差分的同时，在垂直方向做了一次高斯平滑”**，抗噪能力直接拉满！

同理，Sobel Y 滤波器 `[[-1,-2,-1], [0,0,0],[1,2,1]]` 是（下面一行 减去 上面一行），算出来的自然就是垂直方向的颜色变化率啦！